# orbit-ocsp tutorial

One command, three input modes. All of them feed the same scoring engine and
produce the same output files.

| Mode | Input | DE step |
|------|-------|---------|
| `--mode expression` | matrix + groups | yes (R) |
| `--mode genes` | gene symbols / IDs | no |
| `--mode sequence` | KOfam / InterProScan / DeepGOPlus output, or a pre-merged JSON | no |

**Before you start**

```bash
pip install -e ".[notebook]"
orbit-ocsp-download-data --species hsa   # or export ORBIT_OCSP_DATA=/path/to/data
```

Sample inputs live in `examples/data/`, organized by mode.

## Setup

Locate the repo root and confirm the scoring data is reachable.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import json
import os
import subprocess
import sys
from pathlib import Path

# Find the repo root by a landmark that does not depend on the package name.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "examples" / "data").is_dir():
        ROOT = candidate
        break
else:  # pragma: no cover
    raise AssertionError(f"Cannot find the repo root from {Path.cwd()}")

# Works whether or not the package was pip-installed
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "examples" / "data"
OUT = ROOT / "out_tutorial"
OUT.mkdir(exist_ok=True)

SPECIES = "hsa"
CONDITION = "Colorectal Cancer"


def run_cli(*args, check=True):
    """Run the orbit-ocsp CLI and stream its output."""
    cmd = [sys.executable, "-m", "orbit_ocsp", *map(str, args)]
    env = {**os.environ, "PYTHONPATH": str(ROOT)}
    proc = subprocess.run(
        cmd, cwd=ROOT, capture_output=True, text=True, env=env
    )
    out = proc.stdout.strip() or "(no stdout)"
    print(out.replace(str(ROOT) + "/", ""))  # keep paths portable
    if proc.returncode != 0:
        print("--- stderr ---", file=sys.stderr)
        print(proc.stderr.strip()[-2000:], file=sys.stderr)
        if check:
            raise RuntimeError(f"exit {proc.returncode}")
    return None


print(f"repo root: {ROOT.name}/")

repo root: orbit-ocsp/


In [2]:
from orbit_ocsp.data_manager import data_status

status = data_status(SPECIES)
print(f"data root : {Path(status['data_root']).name}/  (resolved locally)")
print(f"ready     : {status['ready']}")
if not status["ready"]:
    print(f"missing   : {status['missing'][:5]}")
    print("\nRun `orbit-ocsp-download-data --species hsa`, or set ORBIT_OCSP_DATA.")
    print("Merge-only sequence steps below still work without it.")

data root : data/  (resolved locally)
ready     : True


In [3]:
# --condition must match the background library exactly.
# Browse the valid values before picking one. The listing is ordered by how
# many background records support each condition, so the best-supported
# choices come first — a condition backed by 3 records gives a much thinner
# background than one backed by 300.
proc = subprocess.run(
    [sys.executable, "-m", "orbit_ocsp.list_b_fields",
     "--species", SPECIES, "--field", "condition", "--top", "15"],
    cwd=ROOT, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(ROOT)},
)
print(proc.stdout.strip())
print()
# Drop --top to see all of them, or --sort alpha to browse by name.


condition (15 values, most records first)
  - 6523  Normal
  -  326  Colorectal Cancer
  -  270  Retinoblastoma
  -  181  Coronavirus disease
  -  167  Pancreatic ductal adenocarcinoma
  -  167  Prostate Cancer
  -  127  Breast Cancer
  -  126  Pancreatic cancer
  -  121  Leber Congenital Amaurosis
  -  100  NGLY1 Deficiency
  -   89  SARS-CoV-2 infection
  -   83  Glioblastoma
  -   68  Ovarian Cancer
  -   56  Autism Spectrum Disorder
  -   55  Embryonal tumor with multilayered rosettes (ETMR)



---
## Mode 1 — Expression matrix

`matrix.tsv` + `groups.tsv` → differential expression → top-k genes →
pathway lookup → ensemble scoring.

- `matrix.tsv`: first column `gene`, remaining columns are sample IDs
- `groups.tsv`: columns `sample_id`, `group` with values `case` / `control`
- Demo data: [GSE50760](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE50760)
  primary colorectal cancer (`case`) vs adjacent normal colon (`control`);
  integer RNA-seq counts for 60 annotated genes × 36 samples. Matches the
  `--condition "Colorectal Cancer"` background used below.


In [ ]:
import pandas as pd

matrix = pd.read_csv(DATA / "expression/matrix.tsv", sep="\t")
groups = pd.read_csv(DATA / "expression/groups.tsv", sep="\t")
print(f"matrix: {matrix.shape[0]} genes x {matrix.shape[1] - 1} samples")
display(matrix.head())
display(groups)

In [ ]:
# The default --de-backend r needs Bioconductor (DESeq2 / limma / edgeR).
# Fall back to the mock backend so this notebook runs anywhere.
import shutil

backend = "r" if shutil.which("Rscript") else "mock"
if backend == "mock":
    print("Rscript not found — using --de-backend mock (synthetic DE numbers).\n")

run_cli(
    "--mode", "expression",
    "--matrix", DATA / "expression/matrix.tsv",
    "--groups", DATA / "expression/groups.tsv",
    "--data-type", "rnaseq_count",  # alias: rnaseq
    "--species", SPECIES,
    "--condition", CONDITION,
    "--de-backend", backend,
    # Default thresholds already clear many genes on this CRC vs normal matrix.
    "--outdir", OUT / "expression",
)


In [ ]:
# DE runs before any filtering, so inspect it to choose sensible thresholds.
de = pd.read_csv(OUT / "expression/de_results.tsv", sep="\t")
print(f"{len(de)} genes tested with engine={de['engine'].iloc[0]}")
print(f"best padj = {de['padj'].min():.3f}")
print(f"genes at the default padj <= 0.05: {(de['padj'] <= 0.05).sum()}")
display(
    de.nsmallest(5, "padj")[["gene", "log2FoldChange", "pvalue", "padj"]]
)

---
## Mode 2 — Gene list

No DE step. Symbols are normalized, pathways looked up in
`data/protein/all_merged_result.json`, then scored.

In [7]:
genes = (DATA / "genes/genes.txt").read_text().split()
print("genes:", genes)

run_cli(
    "--mode", "genes",
    "--genes-file", DATA / "genes/genes.txt",
    "--species", SPECIES,
    "--condition", CONDITION,
    "--outdir", OUT / "genes",
)

genes: ['LEF1', 'CD44', 'LGR5', 'AXIN2']


[genes] ranked 4 genes
Results: out_tutorial/genes/biomarker_ranked.json


In [8]:
ranked_path = OUT / "genes" / "biomarker_ranked.json"
if ranked_path.exists():
    ranked = json.loads(ranked_path.read_text())
    rows = [
        {
            "rank": r["biomarker_rank"],
            "gene": r.get("gene_symbol") or r["query_id"],
            "status": r["scoring_status"],
            "verdict": (r.get("biomarker_score") or {}).get("verdict"),
            "consensus": (r.get("biomarker_score") or {}).get("consensus_score"),
            "primary_p": (r.get("biomarker_score") or {}).get("primary_p_value"),
        }
        for r in ranked
    ]
    display(pd.DataFrame(rows))
else:
    print("No results — scoring data is probably missing.")

,rank,gene,status,verdict,consensus,primary_p
0,1,CD44,ok,enriched,0.6,3.430675e-13
1,2,LEF1,ok,enriched,0.6,1.636262e-12
2,3,AXIN2,ok,enriched,0.2,2.015562e-08
3,4,LGR5,ok,not_sig,0.0,5.297297e-03


---
## Mode 3 — Sequence annotation

orbit-ocsp **does not run** KOfam, InterProScan, DeepGOPlus or BLAST. You run
those yourself; orbit-ocsp parses their native output, merges the terms, and
scores.

Two entry points:

- **A** — raw tool output → parse → merge → score
- **B** — an already-merged A_terms JSON → validate → score

### The three native formats

In [9]:
NATIVE = DATA / "sequence/native"

print("=== KOfam (whitespace-aligned; '#' comments, optional leading '*') ===")
print("\n".join((NATIVE / "kofam/1.txt").read_text().splitlines()[:5]))

print("\n=== InterProScan (15-col TSV, no header; GO in column 14) ===")
cols = (NATIVE / "interproscan/1.tsv").read_text().splitlines()[0].split("\t")
print(f"{len(cols)} columns; col14 (GO) = {cols[13]!r}")
print(f"col15 (pathway xrefs, ignored) = {cols[14][:70]}...")

print("\n=== DeepGOPlus (query_id, GO term, score) ===")
print("\n".join((NATIVE / "deepgoplus/1.tsv").read_text().splitlines()[:5]))

=== KOfam (whitespace-aligned; '#' comments, optional leading '*') ===
# gene name           KO     thrshld  score   E-value KO definition
#-------------------- ------ ------- ------ --------- ---------------------
  NP_570602.2         K26168  316.13  176.3   2.9e-56 immunoglobulin superfamily member 1
  NP_570602.2         K06512  381.60  124.4   2.4e-40 leukocyte immunoglobulin-like receptor
  NP_570602.2         K26357  269.73  101.4   2.9e-33 T-cell-interacting, activating receptor on myeloid cells protein 1

=== InterProScan (15-col TSV, no header; GO in column 14) ===
15 columns; col14 (GO) = '-'
col15 (pathway xrefs, ignored) = MetaCyc:PWY-1901|MetaCyc:PWY-1921|MetaCyc:PWY-1961|MetaCyc:PWY-1981|Me...

=== DeepGOPlus (query_id, GO term, score) ===
NP_570602.2	GO:0001775	0.129
NP_570602.2	GO:0002376	0.214
NP_570602.2	GO:0002682	0.189
NP_570602.2	GO:0002683	0.159
NP_570602.2	GO:0002684	0.110


### Entry A, step 1 — merge only

`--merge-only` builds the A_terms JSON and stops, which is the fastest way to
inspect what each tool contributed. No scoring data required.

In [10]:
run_cli(
    "--mode", "sequence",
    "--kofam", NATIVE / "kofam/1.txt",
    "--interproscan", NATIVE / "interproscan/1.tsv",
    "--deepgo", NATIVE / "deepgoplus/1.tsv",
    "--id-map", NATIVE / "id_map.tsv",
    "--species", SPECIES,
    "--merge-only",
    "--outdir", OUT / "sequence_merge",
)

[sequence] merged 1 record(s)
Merged JSON: out_tutorial/sequence_merge/merged_a_terms.json
Report: out_tutorial/sequence_merge/sequence_merge_report.json


In [11]:
merged = json.loads((OUT / "sequence_merge/merged_a_terms.json").read_text())[0]

print("Contract fields (what the scoring engine reads):")
for field in ("gene_name", "similarity_gene_name", "ENTREZ_ID"):
    print(f"  {field:22} = {merged[field]!r}")
print(f"  {'pathway':22} = {len(merged['pathway'])} terms")

print("\nExtension fields (provenance, ignored by the engine):")
for field in ("pathway_sources", "term_evidence", "id_source", "id_evidence"):
    if field in merged:
        print(f"  {field}")

kegg = [t for t in merged["pathway"] if not t.startswith("GO:")]
print(f"\nKEGG pathways: {len(kegg)}   GO terms: {len(merged['pathway']) - len(kegg)}")
print(f"id_source: {merged['id_source']}  ->  {merged['similarity_gene_name']}")

Contract fields (what the scoring engine reads):
  gene_name              = 'NP_570602.2'
  similarity_gene_name   = 'A1BG'
  ENTREZ_ID              = '1'
  pathway                = 227 terms

Extension fields (provenance, ignored by the engine):
  pathway_sources
  term_evidence
  id_source
  id_evidence

KEGG pathways: 95   GO terms: 132
id_source: id_map  ->  A1BG


In [12]:
# Which tool supplied each term? Terms found by several tools are the strongest.
from collections import Counter

combos = Counter(
    " + ".join(sources) for sources in merged["pathway_sources"].values()
)
display(
    pd.DataFrame(
        sorted(combos.items(), key=lambda kv: -kv[1]),
        columns=["evidence source(s)", "n terms"],
    )
)

multi = [t for t, s in merged["pathway_sources"].items() if len(s) > 1]
print(f"\nTerms supported by more than one tool: {len(multi)}")
print(multi[:5])

,evidence source(s),n terms
0,deepgoplus,130
1,kofam,95
2,interproscan,1
3,interproscan + deepgoplus,1



Terms supported by more than one tool: 1
['GO:0005886']


In [13]:
# The diagnostic report explains every skipped line and excluded record.
report = json.loads((OUT / "sequence_merge/sequence_merge_report.json").read_text())

display(pd.DataFrame([
    {
        "source": i["source"],
        "parsed": i["parsed_lines"],
        "skipped": i["skipped_lines"],
        "filtered": i.get("filtered_lines"),
    }
    for i in report["inputs"]
]))

print("excluded records:", report["excluded"] or "none")
print("warnings        :", report["warnings"] or "none")

,source,parsed,skipped,filtered
0,kofam,77,0,0
1,interproscan,29,0,0
2,deepgoplus,131,0,0


excluded records: none
warnings        : none


### The DeepGOPlus threshold

Default is `0`, i.e. keep every prediction. Raise `--deepgo-min-score` to drop
low-confidence GO terms. Records whose score cannot be parsed are always kept
and never threshold-filtered.

In [14]:
from orbit_ocsp.sequence_annotation import parse_deepgoplus

rows = []
for threshold in (0.0, 0.1, 0.3, 0.5, 0.9):
    terms, _, stats = parse_deepgoplus(
        NATIVE / "deepgoplus/1.tsv", min_score=threshold
    )
    rows.append({
        "--deepgo-min-score": threshold,
        "kept": stats.parsed_lines,
        "filtered": stats.filtered_lines,
    })
display(pd.DataFrame(rows))
print("Default 0 keeps everything, matching the earlier merge script.")

,--deepgo-min-score,kept,filtered
0,0.0,131,0
1,0.1,131,0
2,0.3,28,103
3,0.5,11,120
4,0.9,0,131


Default 0 keeps everything, matching the earlier merge script.


### Entry A, step 2 — score the merged terms

In [15]:
run_cli(
    "--mode", "sequence",
    "--kofam", NATIVE / "kofam/1.txt",
    "--interproscan", NATIVE / "interproscan/1.tsv",
    "--deepgo", NATIVE / "deepgoplus/1.tsv",
    "--id-map", NATIVE / "id_map.tsv",
    "--species", SPECIES,
    "--condition", CONDITION,
    "--outdir", OUT / "sequence_native",
)

[sequence] merged 1 record(s)
Merged JSON: out_tutorial/sequence_native/merged_a_terms.json
Report: out_tutorial/sequence_native/sequence_merge_report.json
Results: out_tutorial/sequence_native/biomarker_ranked.json


### Entry B — a pre-merged JSON

If the merge already happened elsewhere, hand the JSON straight to the scorer.
It is validated first: a missing or non-array `pathway` is a hard error, while
empty pathways and records with no usable key are reported and skipped.

In [16]:
MERGED = DATA / "sequence/merged/merged_result_1.json"
record = json.loads(MERGED.read_text())[0]
print(f"gene_name={record['gene_name']!r} "
      f"similarity_gene_name={record['similarity_gene_name']!r} "
      f"ENTREZ_ID={record['ENTREZ_ID']!r}")
print(f"pathway: {len(record['pathway'])} terms")

run_cli(
    "--mode", "sequence",
    "--merged-json", MERGED,
    "--species", SPECIES,
    "--condition", CONDITION,
    "--outdir", OUT / "sequence_merged",
)

gene_name='NP_570602.2' similarity_gene_name='1' ENTREZ_ID='1'
pathway: 162 terms


[sequence] merged 1 record(s)
Merged JSON: examples/data/sequence/merged/merged_result_1.json
Report: out_tutorial/sequence_merged/sequence_merge_report.json
Results: out_tutorial/sequence_merged/biomarker_ranked.json


### Why the two entries disagree

Both describe the same protein, but the pre-merged file has fewer KEGG
pathways. Its generator collapsed each KO to a single pathway, dropping the
rest; the GO half is identical.

In [17]:
a_terms = set(json.loads(
    (OUT / "sequence_merge/merged_a_terms.json").read_text()
)[0]["pathway"])
b_terms = set(record["pathway"])


def split(terms):
    go = {t for t in terms if t.startswith("GO:")}
    return len(terms - go), len(go)


display(pd.DataFrame([
    dict(zip(("KEGG", "GO"), split(a_terms)), input="entry A (native, merged now)"),
    dict(zip(("KEGG", "GO"), split(b_terms)), input="entry B (pre-merged file)"),
])[["input", "KEGG", "GO"]])

print(f"GO sets identical      : {({t for t in a_terms if t.startswith('GO:')} == {t for t in b_terms if t.startswith('GO:')})}")
print(f"entry B KEGG ⊂ entry A : {({t for t in b_terms if not t.startswith('GO:')} <= {t for t in a_terms if not t.startswith('GO:')})}")
print(f"only in entry A        : {len(a_terms - b_terms)} terms")

,input,KEGG,GO
0,"entry A (native, merged now)",95,132
1,entry B (pre-merged file),30,132


GO sets identical      : True
entry B KEGG ⊂ entry A : True
only in entry A        : 65 terms


### Batch mode

`--annotation-dir` walks `<dir>/kofam/`, `<dir>/interproscan/`,
`<dir>/deepgoplus/` and groups files by filename stem. Every sample is scored
and merged into a single `biomarker_ranked.json` with one cross-sample ranking;
each record keeps its `sample_key`.

In [18]:
run_cli(
    "--mode", "sequence",
    "--annotation-dir", NATIVE,
    "--id-map", NATIVE / "id_map.tsv",
    "--species", SPECIES,
    "--merge-only",
    "--outdir", OUT / "sequence_batch",
)

batch = json.loads((OUT / "sequence_batch/sequence_merge_report.json").read_text())
print("per-sample status:", batch["samples"])

[sequence] merged 1 record(s)
Merged JSON: out_tutorial/sequence_batch/merged_a_terms.json
Report: out_tutorial/sequence_batch/sequence_merge_report.json
per-sample status: {'1': {'status': 'ok', 'n_records': 1}}


---
## Reading the results

Every mode writes the same files. `biomarker_ranked.tsv` is the flat view;
`method_scores.tsv` breaks the ensemble down per method.

In [19]:
for name in ("expression", "genes", "sequence_native", "sequence_merged"):
    d = OUT / name
    files = sorted(p.name for p in d.iterdir()) if d.is_dir() else []
    print(f"{name:18} {files or '(not produced)'}")

expression         ['biomarker_ranked.json', 'biomarker_ranked.tsv', 'de_results.tsv', 'gene_reports', 'method_scores.tsv', 'pipeline_summary.json', 'topk_de_genes.tsv']
genes              ['biomarker_ranked.json', 'biomarker_ranked.tsv', 'gene_reports', 'method_scores.tsv', 'pipeline_summary.json']
sequence_native    ['biomarker_ranked.json', 'biomarker_ranked.tsv', 'gene_reports', 'merged_a_terms.json', 'method_scores.tsv', 'pipeline_summary.json', 'sequence_merge_report.json']
sequence_merged    ['biomarker_ranked.json', 'biomarker_ranked.tsv', 'gene_reports', 'method_scores.tsv', 'pipeline_summary.json', 'sequence_merge_report.json']


In [20]:
# Per-method breakdown for one scored gene.
# effect_size is the standardized deviation from the permutation null, so it
# is comparable across methods; observed_statistic is not.
methods = OUT / "sequence_native" / "method_scores.tsv"
if methods.exists():
    df = pd.read_csv(methods, sep="\t")
    display(df[["gene", "method", "observed_statistic", "p_value",
                "effect_size", "verdict", "consensus_score"]])
else:
    print("Not produced — scoring data is probably missing.")


,gene,method,observed_statistic,p_value,effect_size,verdict,consensus_score
0,A1BG,hypergeometric,99.000000,1.082744e-21,10.934441,enriched,0.2
1,A1BG,resnik_bma,0.722035,1.176471e-01,1.262558,not_sig,0.2
2,A1BG,lin_bma,0.248918,1.372549e-01,1.308451,not_sig,0.2
3,A1BG,jaccard,0.014486,8.208955e-01,-0.795328,not_sig,0.2
4,A1BG,overlap,99.000000,8.208955e-01,-0.795209,not_sig,0.2


---
## Summary

| Mode | Command | Sample data |
|------|---------|-------------|
| Expression | `orbit-ocsp --mode expression --matrix ... --groups ...` | `examples/data/expression/` |
| Gene list | `orbit-ocsp --mode genes --genes-file ...` | `examples/data/genes/` |
| Sequence A | `orbit-ocsp --mode sequence --kofam ... --interproscan ... --deepgo ...` | `examples/data/sequence/native/` |
| Sequence B | `orbit-ocsp --mode sequence --merged-json ...` | `examples/data/sequence/merged/` |

Useful flags: `--merge-only` (build the JSON, skip scoring),
`--annotation-dir` (batch), `--deepgo-min-score` (confidence cutoff),
`--id-map` (attach Entrez IDs / symbols to novel sequences).

Further reading: `docs/SEQUENCE_ANNOTATION.md` (annotation tools and
formats), `docs/OUTPUTS.md` (every output field), `docs/ADVANCED.md`
(background filters, single-method runs).

Clean up with `shutil.rmtree(OUT)`.